# UCS420 – Cognitive Computing | Assignment 10
## Heart Health Information Assistant (Even Roll Number)

**Objective:** Build a Python-based healthcare cognitive assistant that preprocesses user queries, classifies intent, retrieves information from a heart-health knowledge base, applies reasoning rules, and responds through an interactive interface.

**How to use:** Open this notebook in Google Colab (`File → Upload notebook`), then run the cells from top to bottom. The final cell launches an interactive chat-style interface.

> **Safety notice:** This is an educational demonstration, not a medical device. It does not diagnose, prescribe, or replace a qualified clinician. For possible heart-attack or stroke symptoms, call your local emergency number immediately (in India, 112 or 108 where available). Do not wait for this assistant's prediction.


## 1. Install and import libraries

The notebook uses NLTK for text preprocessing and scikit-learn for TF-IDF, classification, and evaluation. In Colab, run this cell first.

In [ ]:
%pip -q install nltk scikit-learn pandas matplotlib seaborn ipywidgets

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
print("Libraries imported successfully.")


## 2. Labelled query corpus

The corpus below contains **70 labelled queries across 7 intent categories** (10 examples per category). The examples are intentionally varied, but this is a small teaching dataset; evaluation scores should not be interpreted as real-world clinical performance.

In [ ]:
training_examples = {
    "symptoms": [
        "What are common warning signs of heart disease?",
        "Why do I feel chest discomfort when walking?",
        "Can shortness of breath be related to the heart?",
        "What does an irregular heartbeat feel like?",
        "I sometimes feel dizzy and my heart races",
        "What are symptoms of poor blood circulation?",
        "Can unusual fatigue indicate a heart problem?",
        "What is angina and how may it feel?",
        "Why do my ankles swell sometimes?",
        "What symptoms should make me discuss heart health with a doctor?"
    ],
    "risk_factors": [
        "Does high blood pressure increase heart disease risk?",
        "How does smoking affect the heart?",
        "Is family history a risk factor for heart disease?",
        "Can diabetes increase cardiovascular risk?",
        "Does high cholesterol affect arteries?",
        "How does obesity influence heart health?",
        "Can stress contribute to heart problems?",
        "Does age change my risk of cardiovascular disease?",
        "Is physical inactivity bad for the heart?",
        "What are the major risk factors for a heart attack?"
    ],
    "prevention": [
        "How can I keep my heart healthy?",
        "What lifestyle changes help prevent heart disease?",
        "How can I reduce my cardiovascular risk?",
        "What habits protect the heart over time?",
        "How often should I get a routine health check?",
        "What can I do to improve my heart health naturally?",
        "How important is quitting tobacco for prevention?",
        "Can better sleep support heart health?",
        "How do I build a heart healthy daily routine?",
        "What steps can help prevent high blood pressure?"
    ],
    "nutrition": [
        "What foods are good for heart health?",
        "Suggest a heart healthy diet",
        "Should I reduce salt in my meals?",
        "Are fruits and vegetables helpful for the heart?",
        "What fats should I limit for cardiovascular health?",
        "Is eating whole grains good for my heart?",
        "How can I reduce saturated fat in my diet?",
        "What are examples of low sodium foods?",
        "Does eating fish support heart health?",
        "What should I know about sugary drinks and heart health?"
    ],
    "exercise": [
        "How much exercise is good for the heart?",
        "Is brisk walking beneficial for cardiovascular health?",
        "Can regular cycling improve heart fitness?",
        "What physical activities support a healthy heart?",
        "How should a beginner start exercising safely?",
        "Does sitting for long periods affect heart health?",
        "Are strength exercises useful for overall heart health?",
        "How many days a week should adults be active?",
        "Can swimming be a heart healthy activity?",
        "Should I ask my doctor before starting intense exercise?"
    ],
    "monitoring": [
        "What is a normal blood pressure reading for adults?",
        "How can I monitor my blood pressure at home?",
        "What does a cholesterol test measure?",
        "What is resting heart rate?",
        "Why might a doctor recommend an ECG?",
        "What numbers are measured in a lipid profile?",
        "How often should blood pressure be checked?",
        "What is the purpose of a heart rate monitor?",
        "What does systolic blood pressure mean?",
        "What does diastolic blood pressure mean?"
    ],
    "medication_care": [
        "Why might a doctor prescribe medicine for blood pressure?",
        "Should I stop my heart medicine if I feel better?",
        "Can I change my medication dose myself?",
        "What should I ask my doctor about heart medication?",
        "Why is it important to take prescribed medicine regularly?",
        "Can a pharmacist explain possible medicine side effects?",
        "What should I do if I miss a prescribed dose?",
        "Can supplements replace prescribed heart medicines?",
        "Should I tell my doctor about all medicines I take?",
        "How can I keep track of my prescribed medicines?"
    ]
}

rows = [
    {"query": query, "intent": intent}
    for intent, queries in training_examples.items()
    for query in queries
]
corpus = pd.DataFrame(rows)
print(f"Total labelled queries: {len(corpus)}")
print(f"Intent categories: {corpus['intent'].nunique()}")
display(corpus.groupby("intent").size().rename("number_of_queries").to_frame())
display(corpus.head(10))


## 3. Hierarchical intent taxonomy

The taxonomy has **4 major categories**, each with at least 2 subcategories. The classifier uses the seven leaf intents represented in the labelled corpus. The taxonomy also shows how those intents fit into a broader healthcare information structure.

In [ ]:
taxonomy = {
    "Heart health awareness": {
        "Symptoms and warning signs": ["symptoms", "possible warning signs", "when to seek care"],
        "Risk and causes": ["risk_factors", "modifiable risks", "non-modifiable risks"]
    },
    "Healthy lifestyle": {
        "Food and nutrition": ["nutrition", "sodium", "healthy fats and fibre"],
        "Physical activity": ["exercise", "aerobic activity", "strength and movement"]
    },
    "Clinical care": {
        "Health measurements": ["monitoring", "blood pressure", "heart rate and cholesterol"],
        "Medication support": ["medication_care", "safe medicine use", "clinician discussion"]
    },
    "Prevention and follow-up": {
        "Risk reduction": ["prevention", "tobacco avoidance", "sleep and stress habits"],
        "Ongoing care": ["routine check-ups", "follow-up", "shared decision-making"]
    }
}

for major, subcats in taxonomy.items():
    print(major)
    for subcategory, examples in subcats.items():
        print(f"  ├─ {subcategory}: {', '.join(examples)}")


## 4. Domain concepts and relationships

The domain model contains more than 8 concepts and more than 10 directed relationships. These are simplified educational relationships, not a complete clinical ontology.

In [ ]:
domain_concepts = {
    "Heart": "Organ that circulates blood through the body",
    "Blood pressure": "Force of blood against artery walls",
    "Cholesterol": "A blood lipid measured as part of cardiovascular risk assessment",
    "Arteries": "Blood vessels that carry blood away from the heart",
    "Smoking": "Tobacco exposure that increases cardiovascular risk",
    "Physical activity": "Movement and exercise that support cardiovascular fitness",
    "Nutrition": "Dietary pattern influencing cardiovascular risk",
    "Diabetes": "A condition associated with increased cardiovascular risk",
    "Medication": "Treatment prescribed by a qualified healthcare professional",
    "ECG": "Test that records the electrical activity of the heart",
    "Symptoms": "Experiences such as chest discomfort or breathlessness that need context",
    "Clinician": "Qualified health professional who assesses and advises a patient"
}

relationships = [
    ("Smoking", "increases risk of", "Heart disease"),
    ("Diabetes", "increases risk of", "Heart disease"),
    ("High blood pressure", "can damage", "Arteries"),
    ("High cholesterol", "can contribute to plaque in", "Arteries"),
    ("Physical activity", "supports", "Heart health"),
    ("Heart healthy nutrition", "supports", "Heart health"),
    ("Clinician", "may recommend", "ECG"),
    ("ECG", "records electrical activity of", "Heart"),
    ("Clinician", "prescribes when appropriate", "Medication"),
    ("Medication", "may help manage", "Blood pressure"),
    ("Symptoms", "should be assessed by", "Clinician"),
    ("Blood pressure monitoring", "helps track", "Blood pressure")
]

print("Domain concepts:")
for concept, description in domain_concepts.items():
    print(f"• {concept}: {description}")
print(f"\nNumber of concepts: {len(domain_concepts)}")
print("\nRelationships:")
for source, relation, target in relationships:
    print(f"• {source} --{relation}--> {target}")
print(f"\nNumber of relationships: {len(relationships)}")


## 5. NLP preprocessing

The preprocessing function performs the four required steps:
1. Lowercase conversion
2. Tokenization
3. Stopword and punctuation removal
4. Lemmatization

A small fallback is included in case an NLTK tokenizer resource is unavailable.

In [ ]:
try:
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    STOP_WORDS = set()

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    try:
        tokens = word_tokenize(text)
    except LookupError:
        tokens = re.findall(r"[a-z]+", text)
    cleaned = []
    for token in tokens:
        if token.isalpha() and token not in STOP_WORDS:
            try:
                token = lemmatizer.lemmatize(token)
            except LookupError:
                pass
            cleaned.append(token)
    return " ".join(cleaned)

sample = "How can I monitor my Blood Pressure safely?"
print("Original:", sample)
print("Preprocessed:", preprocess_text(sample))


## 6. TF-IDF feature extraction and intent classifier

A scikit-learn pipeline applies the custom preprocessing function, converts text into TF-IDF features, and trains a Logistic Regression classifier. A stratified hold-out set is used for evaluation.

In [ ]:
X = corpus["query"]
y = corpus["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        preprocessor=preprocess_text,
        tokenizer=str.split,
        token_pattern=None,
        ngram_range=(1, 2),
        min_df=1
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Model training complete.")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 7. Model evaluation

The assignment asks for accuracy, precision, recall, F1-score, and a confusion matrix. Macro-averaged precision, recall, and F1 give each intent equal weight.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1-score"],
    "Score": [accuracy, precision, recall, f1]
})
display(metrics.style.format({"Score": "{:.3f}"}))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(corpus["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(9, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted intent")
plt.ylabel("Actual intent")
plt.title("Confusion Matrix – Heart Health Intent Classifier")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Domain-specific knowledge base

The knowledge base provides general information for each intent. It is intentionally cautious: it does not provide individual diagnoses, medication doses, or personalized treatment plans.

In [ ]:
knowledge_base = {
    "symptoms": {
        "title": "Symptoms and warning signs",
        "response": (
            "Heart-related problems can have different symptoms. Possible warning signs include "
            "chest pressure or discomfort, breathlessness, unusual fatigue, dizziness, palpitations, "
            "or swelling. These symptoms can have many causes and cannot be diagnosed by a chatbot. "
            "Seek professional assessment, especially if symptoms are new, worsening, or concerning."
        )
    },
    "risk_factors": {
        "title": "Cardiovascular risk factors",
        "response": (
            "Important cardiovascular risk factors include high blood pressure, abnormal cholesterol, "
            "diabetes, tobacco use, physical inactivity, and some family-history and age-related factors. "
            "A healthcare professional can help assess your overall risk using your history and measurements."
        )
    },
    "prevention": {
        "title": "Heart-health prevention",
        "response": (
            "Helpful general habits include avoiding tobacco, being physically active, choosing a balanced "
            "diet, getting adequate sleep, and attending recommended health check-ups. Changes should be "
            "realistic and appropriate to your health and abilities."
        )
    },
    "nutrition": {
        "title": "Nutrition and heart health",
        "response": (
            "A heart-supportive eating pattern commonly emphasizes vegetables, fruits, whole grains, "
            "beans, nuts, and suitable sources of protein. Limiting excess sodium, highly processed foods, "
            "added sugars, and saturated fats may be helpful. Individual dietary needs can differ."
        )
    },
    "exercise": {
        "title": "Physical activity",
        "response": (
            "Regular movement supports cardiovascular fitness. Walking, cycling, swimming, and other "
            "activities may be options depending on ability and health. Increase activity gradually. "
            "If you have symptoms, a known heart condition, or plan vigorous exercise after inactivity, "
            "ask a healthcare professional what is safe for you."
        )
    },
    "monitoring": {
        "title": "Heart-health measurements",
        "response": (
            "Blood pressure, cholesterol tests, and sometimes heart-rate measurements help clinicians "
            "understand cardiovascular health. An ECG records the heart's electrical activity. Results "
            "should be interpreted in context by a qualified professional; one reading alone may not tell "
            "the whole story."
        )
    },
    "medication_care": {
        "title": "Medication and clinical care",
        "response": (
            "Take medicines only as prescribed. Do not start, stop, or change a dose based on chatbot "
            "advice. If you miss a dose, notice a possible side effect, or have questions, consult your "
            "prescriber or pharmacist and follow the medicine's official instructions."
        )
    }
}

for intent, item in knowledge_base.items():
    print(f"{intent}: {item['title']}")


## 9. Reasoning rules

The assistant applies explicit rules in addition to the ML prediction:
- Emergency escalation has the highest priority.
- Low-confidence predictions are not presented as reliable classifications.
- Unsupported queries are redirected.
- Medication changes are always referred to a clinician.
- A confidence score is displayed for transparency.
- General information includes a reminder that the assistant is not a diagnostic tool.

Emergency keyword matching is deliberately conservative but cannot detect every emergency. Never rely on this assistant to rule out an emergency.

In [ ]:
EMERGENCY_KEYWORDS = [
    "crushing chest pain", "severe chest pain", "chest pain", "chest pressure",
    "cannot breathe", "can't breathe", "severe shortness of breath",
    "shortness of breath with chest pain", "fainting", "passed out",
    "face drooping", "arm weakness", "slurred speech", "stroke symptoms"
]

SUPPORTED_HINTS = [
    "heart", "cardiac", "blood pressure", "cholesterol", "pulse", "ecg",
    "heart rate", "artery", "arteries", "cardiovascular", "chest",
    "breath", "smoking", "diabetes", "exercise", "diet", "salt",
    "medication", "medicine", "symptom", "angina", "ticker", "flutter", "fluttering", "palpitations"
]

LOW_CONFIDENCE_THRESHOLD = 0.35

def is_emergency(query):
    q = query.lower()
    return any(keyword in q for keyword in EMERGENCY_KEYWORDS)

def appears_heart_related(query):
    q = query.lower()
    return any(term in q for term in SUPPORTED_HINTS)

def generate_response(query):
    query = str(query).strip()
    if not query:
        return {
            "intent": "not classified", "confidence": 0.0,
            "response": "Please enter a question about heart health."
        }

    # Rule 1: emergency escalation overrides model classification.
    if is_emergency(query):
        return {
            "intent": "EMERGENCY – immediate escalation",
            "confidence": None,
            "response": (
                "EMERGENCY WARNING: Chest pain, severe breathing difficulty, fainting, or stroke-like "
                "symptoms may require urgent medical attention. Call your local emergency number now "
                "(in India, 112 or 108 where available). Do not drive yourself if you may be having an "
                "emergency. Do not wait for an online response."
            )
        }

    # Rule 2: unsupported-topic handling.
    if not appears_heart_related(query):
        return {
            "intent": "unsupported_query", "confidence": None,
            "response": (
                "I am designed for general heart-health information, so I cannot answer this topic. "
                "Please ask about heart-health symptoms, risk factors, prevention, nutrition, exercise, "
                "monitoring, or medication safety."
            )
        }

    probabilities = model.predict_proba([query])[0]
    best_index = int(np.argmax(probabilities))
    predicted_intent = model.classes_[best_index]
    confidence = float(probabilities[best_index])

    # Rule 3: low-confidence handling.
    if confidence < LOW_CONFIDENCE_THRESHOLD:
        return {
            "intent": "low_confidence",
            "confidence": confidence,
            "response": (
                "I am not sufficiently confident about the category of this question. "
                "Could you rephrase it or make the heart-health topic more specific? "
                "For personal medical concerns, contact a qualified healthcare professional."
            )
        }

    # Rule 4: medication safety override.
    if predicted_intent == "medication_care" and any(
        word in query.lower() for word in ["stop", "change", "increase", "decrease", "dose", "replace"]
    ):
        extra = (
            " Important: do not change or stop a prescribed medicine without advice from your "
            "prescriber or pharmacist."
        )
    else:
        extra = ""

    # Rule 5: knowledge retrieval and safe response.
    response = knowledge_base[predicted_intent]["response"] + extra
    response += "\n\nThis is general educational information, not a diagnosis or personal treatment plan."

    return {
        "intent": predicted_intent,
        "confidence": confidence,
        "response": response
    }

print("Reasoning engine ready.")


## 10. Integration: NLP + classification + reasoning + knowledge retrieval

The function below combines all system components into one assistant. It can be used for single queries or multiple queries.

In [ ]:
def heart_health_assistant(query):
    result = generate_response(query)
    print("=" * 75)
    print("USER QUERY:", query)
    print("DETECTED INTENT:", result["intent"])
    if result["confidence"] is not None:
        print(f"MODEL CONFIDENCE: {result['confidence']:.3f}")
    print("\nASSISTANT RESPONSE:")
    print(result["response"])
    print("=" * 75)
    return result

# Example query
example_result = heart_health_assistant("What foods are good for heart health?")


## 11. Required testing scenarios

The assignment requests tests for each defined intent, a differently worded query, an unsupported healthcare query, a low-confidence query, and an emergency query. The tests below cover all seven intents and the additional cases.

**Note:** The low-confidence example is intentionally vague but contains a heart-related keyword. Its actual confidence depends on the trained model; the cell reports whether the low-confidence rule triggered. You can adjust the threshold for experimentation, but should explain the chosen threshold in your report.

In [ ]:
test_queries = [
    ("Intent: symptoms", "Can heart trouble cause dizziness and breathlessness?"),
    ("Intent: risk factors", "Does tobacco use raise my cardiovascular risk?"),
    ("Intent: prevention", "What daily habits can protect my heart?"),
    ("Intent: nutrition", "Which foods should I choose for a healthier heart?"),
    ("Intent: exercise", "Is walking regularly useful for cardiac fitness?"),
    ("Intent: monitoring", "Why should I measure my blood pressure?"),
    ("Intent: medication care", "Should I stop taking my prescribed heart medicine?"),
    ("Differently worded query", "My ticker sometimes flutters; what could that mean?"),
    ("Unsupported query", "How do I fix a leaking kitchen tap?"),
    ("Low-confidence candidate", "heart xyz"),
    ("Emergency query", "I have severe chest pain and cannot breathe")
]

test_results = []
for scenario, query in test_queries:
    result = generate_response(query)
    test_results.append({
        "Scenario": scenario,
        "Query": query,
        "Detected intent": result["intent"],
        "Confidence": result["confidence"],
        "Response": result["response"]
    })

test_df = pd.DataFrame(test_results)
display(test_df[["Scenario", "Query", "Detected intent", "Confidence"]])


### Inspect full responses for all test cases

In [ ]:
for row in test_results:
    print("\n" + "#" * 90)
    print(row["Scenario"], "|", row["Query"])
    print("Detected intent:", row["Detected intent"])
    if row["Confidence"] is not None:
        print("Confidence:", round(row["Confidence"], 3))
    print("Response:", row["Response"])


## 12. Interactive assistant interface (Google Colab)

Run this cell to create an interactive interface. Enter a question and click **Ask**. The interface also supports repeated queries during the same session.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

query_box = widgets.Textarea(
    value="",
    placeholder="Type a heart-health question here...",
    description="Question:",
    layout=widgets.Layout(width="100%", height="90px")
)
ask_button = widgets.Button(
    description="Ask the Heart Health Assistant",
    button_style="primary",
    icon="heart"
)
output_area = widgets.Output()

def on_ask_clicked(_):
    with output_area:
        clear_output()
        result = generate_response(query_box.value)
        display(Markdown(f"### Detected intent: `{result['intent']}`"))
        if result["confidence"] is not None:
            display(Markdown(f"**Model confidence:** `{result['confidence']:.3f}`"))
        display(Markdown(result["response"]))

ask_button.on_click(on_ask_clicked)
display(widgets.VBox([query_box, ask_button, output_area]))


## 13. Optional: try several queries in one batch

This cell is useful for demonstrating that the assistant accepts multiple queries. Edit the list and run again.

In [ ]:
batch_queries = [
    "How can I reduce my risk of heart disease?",
    "Is cycling good for heart health?",
    "What does an ECG test do?",
    "Can I change my blood pressure medicine dose myself?",
    "I have crushing chest pain"
]

batch_output = []
for q in batch_queries:
    r = generate_response(q)
    batch_output.append({
        "Query": q,
        "Intent": r["intent"],
        "Confidence": r["confidence"],
        "Response": r["response"]
    })

display(pd.DataFrame(batch_output)[["Query", "Intent", "Confidence"]])


## 14. System architecture and workflow

**Input query → emergency/unsupported rules → text preprocessing → TF-IDF vectorization → Logistic Regression intent classification → confidence and safety rules → knowledge-base retrieval → response.**

The emergency rule is checked before ML classification so a high-risk phrase is escalated without relying on the classifier.

### Requirement checklist

| Assignment requirement | Implemented in |
|---|---|
| 40+ labelled queries, 5+ intents | Section 2 (70 queries, 7 intents) |
| 4+ major taxonomy categories, 2+ subcategories each | Section 3 |
| 8+ concepts and 10+ relationships | Section 4 |
| Lowercase, tokenization, stopword removal, lemmatization | Section 5 |
| TF-IDF and ML classifier | Section 6 |
| Accuracy, precision, recall, F1, confusion matrix | Section 7 |
| Domain-specific knowledge base | Section 8 |
| 5+ reasoning rules | Section 9 |
| Integrated assistant | Section 10 |
| Required test scenarios | Section 11 |
| Interactive multi-query interface | Sections 12–13 |

### Conclusion

The notebook demonstrates a complete educational heart-health cognitive assistant. It combines NLP preprocessing, supervised intent classification, confidence-based handling, rule-based safety escalation, and retrieval from a domain-specific knowledge base. Its small labelled corpus is suitable for demonstrating the assignment workflow, but the model is not clinically validated and must not be used to make medical decisions.

### References for further reading
- Scikit-learn documentation: https://scikit-learn.org/stable/
- NLTK documentation: https://www.nltk.org/
- World Health Organization – Cardiovascular diseases: https://www.who.int/health-topics/cardiovascular-diseases
- American Heart Association – Heart health information: https://www.heart.org/
